# 2D Pose Model Comparison

Comparing YOLO11-pose and YOLO26-pose on rondo training drill footage.

**Goal:** assess keypoint quality on leg joints (hips, knees, ankles, feet) across both models and across mixed camera angles. Output is annotated frames and a qualitative decision on which model to use going forward.

**Models compared:**
- `yolo11m-pose`
- `yolo26m-pose`

Both are run at medium size (m) for a fair comparison.

## 0. Setup

In [ ]:
# Install dependencies (Colab only)
%pip install ultralytics supervision -q

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
from google.colab import drive
from typing import List, Any, Dict

drive.mount('/content/drive')

# COCO keypoint indices relevant to kick detection
LOWERKEYPOINTS = {
    "left_hip": 11,
    "right_hip": 12,
    "left_knee": 13,
    "right_knee": 14,
    "left_ankle": 15,
    "right_ankle": 16,
}

# minimum keypoint confidence for display
CONFTHRESHOLD = 0.5

## 1. Upload Video

In [ ]:
videoPath = "/content/drive/MyDrive/football-kick-tracker/data/raw/training_drills_videos/training-drill-1.mp4"
print(f"Loaded: {videoPath}")

# get clip stats
cap = cv2.VideoCapture(videoPath)
fps = cap.get(cv2.CAP_PROP_FPS)
totalFrames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

print(f"Resolution: {width}x{height}")
print(f"FPS: {fps:.1f}")
print(f"Frames: {totalFrames}")
print(f"Duration: {totalFrames / fps:.1f}s")

## 2. Load Models

Load YOLO11m-pose and YOLO26m-pose (YOLO11n-pose as a fallback to allow for comparison) models

In [ ]:
model11m = YOLO("yolo11m-pose.pt")
print("YOLO11m-pose loaded")

try:
    modelCompare = YOLO("yolo26m-pose.pt") # load YOLO26m if available
    print("YOLO26m-pose loaded")
    compareModelName = "YOLO26m-pose"
except Exception as e:
    print(f"YOLO26 not available: {e}")
    print("Falling back to YOLO11n-pose for comparison")
    modelCompare = YOLO("yolo11n-pose.pt") # fallback to YOLO11n for comparison
    compareModelName = "YOLO11n-pose"

## 3. Extract Sample Frames

Extract a spread of frames from across the clip for comparison.

In [ ]:
NFRAMES = 16 # number of sample frames

cap = cv2.VideoCapture(videoPath)
frameIndices = np.linspace(0, totalFrames - 1, NFRAMES, dtype=int)
frames = []

for idx in frameIndices:
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, frame = cap.read()
    if ret:
        frames.append((idx, cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))

cap.release()
print(f"Extracted {len(frames)} frames at indices: {frameIndices}")

## 4. Inference Utilities

In [ ]:
def drawLowerKeypoints(frame: np.ndarray, 
                       results: List[Any], 
                       confThreshold: float=CONFTHRESHOLD) -> np.ndarray:
    img = frame.copy()
    if results[0].keypoints is None:
        return img
    
    kps = results[0].keypoints.data

    colours = {
        "left_hip": (255, 100, 0),
        "right_hip": (255, 100, 0),
        "left_knee": (0, 200, 255),
        "right_knee": (0, 200, 255),
        "left_ankle": (0, 255, 100),
        "right_ankle": (0, 255, 100)
    }

    skeletonPairs = [
        ("left_hip", "left_knee"),
        ("left_knee", "left_ankle"),
        ("right_hip", "right_knee"),
        ("right_knee", "right_ankle"),
    ]

    for person in kps:
        for part, idx in LOWERKEYPOINTS.items():
            x, y, conf = person[idx]
            if conf < confThreshold:
                break
            cv2.circle(img, 
                       (int(x), int(y)), 
                       5, 
                       colours[part], 
                       -1)
            cv2.putText(img, 
                        f"{conf:.2f}",
                        (int(x) + 6, int(y) - 6),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.35,
                        colours[part],
                        1)
            
        for a, b in skeletonPairs:
            xa, ya, ca = person[LOWERKEYPOINTS[a]]
            xb, yb, cb = person[LOWERKEYPOINTS[b]]
            if ca < confThreshold or cb < confThreshold:
                continue
            cv2.line(img,
                    (int(xa), int(ya)), 
                    (int(xb), int(yb)),
                    (200, 200, 200),
                    2)
    return img

def runAndAnnotate(model: YOLO,
                   frames: List[np.ndarray],
                   confThreshold: float=CONFTHRESHOLD) -> List[np.ndarray]:
    annotated = []
    for idx, frame in frames:
        bgr = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
        results = model(bgr, verbose=False)
        annotatedFrame = drawLowerKeypoints(frame,
                                            results,
                                            confThreshold)
        annotated.append((idx, annotatedFrame))
    return annotated

## 5. Run inference On Both Models

In [ ]:
print("Running YOLO11m-pose...")
results11m = runAndAnnotate(model11m, frames)

print(f"Running {compareModelName}...")
resultsCompare = runAndAnnotate(modelCompare, frames)

print("Done.")

## 6. Side-by-Side Comparison

In [ ]:
def plotComparison(resA: List[np.ndarray], 
                   resB: List[np.ndarray],
                   nameA: str="YOLO11m-pose",
                   nameB: str="YOLO26m-pose",
                   nsamples: int=4) -> plt.Figure:
    fig, axes = plt.subplots(nsamples, 2, figsize=(14, nsamples * 4))

    for i in range(min(nsamples, len(resA))):
        idxA, frameA = resA[i]
        idxB, frameB = resB[i]

        axes[i, 0].imshow(frameA)
        axes[i, 0].set_title(f"{nameA} - frame {idxA}")
        axes[i, 0].axis("off")

        axes[i, 1].imshow(frameB)
        axes[i, 1].set_title(f"{nameB} - frame {idxB}")
        axes[i, 1].axis("off")

    plt.suptitle("Leg Keypoint Comparison (orange=hip, blue=knee, green=ankle)", y=1.01)
    plt.tight_layout()
    plt.show()

    return fig

fig = plotComparison(resA=results11m, resB=resultsCompare, nameA="YOLO11m-pose", nameB=compareModelName)

## 7. Confidence Score Summary

In [ ]:
def keypointConfidenceSummary(model: YOLO,
                              frames: List[np.ndarray],
                              label: str,
                              keypointIndexes: Dict[str, int]=LOWERKEYPOINTS):
    capData = {name: [] for name in keypointIndexes}

    for _, frame in frames:
        bgr = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
        results = model(bgr, verbose=False)

        if results[0].keypoints is None:
            continue

        kps = results[0].keypoints.data
        for person in kps:
            for name, idx in keypointIndexes.items():
                conf = float(person[idx][2])
                capData[name].append(conf)
        
    print(f"\n{label} - mean keypoint confidence:")
    for name, confs in capData.items():
        if confs:
            print(f"  {name:<15} {np.mean(confs):.3f} (n={len(confs)})")
        else:
            print(f"  {name:<15} no detections")
    
keypointConfidenceSummary(model=model11m, frames=frames, label="YOLO11m-pose", keypointIndexes=LOWERKEYPOINTS)
keypointConfidenceSummary(model=modelCompare, frames=frames, label=compareModelName, keypointIndexes=LOWERKEYPOINTS)

## 8. Observations

**Things to assess per model:**
- Are leg keypoints detected consistently across all players in frame?
- Do keypoints hold up when players overlap or occlude each other?
- Are ankle keypoints reliable (these are most critical for kick detection)?
- Any systematic failures at a particular camera angle?

**Notes:**
- Tested on a single low res rondo drill clip, 16/924 sampled frames
- YOLO11m detected 128 person instances vs YOLO26m's 57 (did not test YOLO11n). YOLO26m is missing a significant proportion of people
- YOLO11m ankle confidence (~0.94) meaningfully higher than YOLO26m(~0.85)
- YOLO26m confidence scores are consistently lower across all leg joints
- Both models degrade at ankle level relative to hip (expected, but worth monitoring)
- Footage quality and angle variety was limited in this test

**Further experimentation needed before final decision:**
- Run on all 6 rondo clips (maybe gather some other non-rondo clips aswell)
- Increase sampled frames (ideally to full clip)
- Broaden model comparison: YOLO11(n)(s)(l)-pose, ViTPose, RTMPose

**Decision:**
- YOLO11m is the working default pending further experimentation (Further experimentation performed at `notebooks/pose/YOLO_candidate_comparison.ipynb`)